In [ ]:
import os

#!git clone https://github.com/recsyspolimi/RecSys_Course_AT_PoliMi

!pwd
os.chdir("../RecSys_Course_AT_PoliMi")
!pwd

#!python run_compile_all_cython.py

In [ ]:
import os
import time 
import numpy as np
import pandas as pd
import scipy.sparse as sp
import scipy.sparse as sps
import matplotlib.pyplot as pyplot
%matplotlib inline

from sklearn.model_selection import KFold
from Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from skopt.space import Real, Integer, Categorical
from scipy.sparse import csr_matrix
from Evaluation.Evaluator import EvaluatorHoldout
from HyperparameterTuning.SearchBayesianSkopt import SearchBayesianSkopt

from Recommenders.SLIM.SLIMElasticNetRecommender import SLIMElasticNetRecommender
from Recommenders.EASE_R.EASE_R_Recommender import EASE_R_Recommender
from Recommenders.GraphBased.RP3betaRecommender import RP3betaRecommender
from Recommenders.KNN.ItemKNNCBFRecommender import ItemKNNCBFRecommender
from Recommenders.KNN.ItemKNNCFRecommender import ItemKNNCFRecommender
from Recommenders.KNN.ItemKNNCustomSimilarityRecommender import ItemKNNCustomSimilarityRecommender

#from Recommenders.MatrixFactorization.IALSRecommender import FeatureCombinedImplicitALSRecommender

# from Recommenders.MatrixFactorization.IALSRecommender import IALSRecommender

#from Recommenders.hybrid.m2_model import IntegratedHierarchicalHybridRecommender
from Recommenders.hybrid.SimilarityMergingHybridRecommender import SimilarityMergingHybridRecommender
from Recommenders.hybrid.LinearHybridRecommender import GeneralizedLinearCoupleHybridRecommender


In [ ]:
df_train = pd.read_csv("data_train.csv")
df_test_user = pd.read_csv("data_target_users_test.csv")

# Training knn costum similarity

In [ ]:
SLIM_params = {
    'topK': 625,
    'l1_ratio': 0.09517221375634205,
    'alpha': 0.00228698730766055
}

"""KNN_params = {
    'similarity': 'tversky',
    'topK': 8,
    'shrink': 100,
    'tversky_alpha': 0.18445514996044549,
    'tversky_beta': 1.7490566752549062,
    'feature_weighting': 'TF-IDF',
}"""

"""IALS_params = {
    'iterations': 135,
    'factors': 87,
    'alpha': 7.762338288061237,
    'regularization': 0.004799745261257595
}"""

EASE_params = {
    'topK': 1431,
    'l2_norm': 426.57622242296605
}

rp3_params = {
    'alpha': 0.7733352330682174,
    'beta': 0.4139018623121251,
    'topK': 35
}

In [ ]:
from scipy.sparse import coo_matrix

#valore 1 per ogni coppia (row, col)
data = [1] * len(df_train)
df_train["row"] = df_train["row"].astype(int)
df_train["col"] = df_train["col"].astype(int)

# matrice COO
URM_all = sp.csr_matrix((data, (df_train["row"], df_train["col"])))


In [ ]:
def split_train_in_five_percentage_global_sample(URM_all, train_percentages):
    """
    The function splits an URM in five matrices based on provided percentages.
    :param URM_all: The full URM matrix
    :param train_percentages: A list of percentages (must sum to 1.0)
    :return: A list of 5 sparse matrices
    """

    import numpy as np
    from scipy.sparse import coo_matrix
    from Data_manager.IncrementalSparseMatrix import IncrementalSparseMatrix

    assert len(train_percentages) == 5, "You must provide exactly 5 percentages."
    assert abs(sum(train_percentages) - 1.0) < 1e-6, "Percentages must sum to 1.0."

    num_users, num_items = URM_all.shape

    # Builders for each of the 5 matrices
    builders = [
        IncrementalSparseMatrix(n_rows=num_users, n_cols=num_items, auto_create_col_mapper=False, auto_create_row_mapper=False)
        for _ in range(5)
    ]

    URM_all_coo = coo_matrix(URM_all)

    # Shuffle indices
    indices_for_sampling = np.arange(URM_all.nnz, dtype=np.int32)
    np.random.shuffle(indices_for_sampling)

    # Calculate the number of interactions for each split
    split_sizes = [int(URM_all.nnz * percentage) for percentage in train_percentages]
    cumulative_sizes = np.cumsum(split_sizes)

    # Divide the indices into 5 groups
    indices_splits = [
        indices_for_sampling[cumulative_sizes[i - 1]:cumulative_sizes[i]] if i > 0 else indices_for_sampling[:cumulative_sizes[i]]
        for i in range(5)
    ]

    # Populate the builders
    for i, builder in enumerate(builders):
        builder.add_data_lists(
            URM_all_coo.row[indices_splits[i]],
            URM_all_coo.col[indices_splits[i]],
            URM_all_coo.data[indices_splits[i]],
        )

    # Convert to sparse matrices
    sparse_matrices = [builder.get_SparseMatrix() for builder in builders]

    # Ensure all outputs are in csr_matrix format
    sparse_matrices = [sp.csr_matrix(matrix) for matrix in sparse_matrices]

    return sparse_matrices

In [ ]:
train_percentages = [0.2, 0.2, 0.2, 0.2, 0.2]  # Cinque parti uguali

URM_parts = split_train_in_five_percentage_global_sample(URM_all, train_percentages)
URM_parts

In [ ]:
import os
from scipy import sparse

output_folder = "./saved_models/"
urm_folder = "./saved_urm/"

for folder in [output_folder, urm_folder]:
    if not os.path.exists(folder):
        os.makedirs(folder)

prefitted_folds = []

for i in range(5):
    print(f"Fitting fold {i+1}/5...")
    
    urm_path = os.path.join(urm_folder, f"URM_train_fold_{i}.npz")
    
    # 1. Gestione URM Train
    if os.path.exists(urm_path):
        print(f"Loading URM_train for fold {i}...")
        URM_train = sparse.load_npz(urm_path)
    else:
        print(f"Creating and saving URM_train for fold {i}...")
        URM_train = sum(URM_parts[j] for j in range(len(URM_parts)) if j != i)
        sparse.save_npz(urm_path, URM_train)

    URM_test = URM_parts[i]
    evaluator_test = EvaluatorHoldout(URM_test, cutoff_list=[20])

    # 2. Inizializzazione Modelli
    recommender_slim = SLIMElasticNetRecommender(URM_train)
    recommender_ease = EASE_R_Recommender(URM_train)
    recommender_rp3 = RP3betaRecommender(URM_train)

    model_names = {
        "slim": (recommender_slim, SLIM_params),
        "ease": (recommender_ease, EASE_params),
        "rp3": (recommender_rp3, rp3_params)
    }

    # 3. Fit o Load dei modelli
    for name, (model, params) in model_names.items():
        file_name = f"{name}_fold_{i}"
        # Verifichiamo se il file del modello esiste (il framework aggiunge solitamente un'estensione o crea una cartella)
        try:
            model.load_model(output_folder, file_name=file_name)
            print(f"Loaded {name} from disk.")
        except (FileNotFoundError, Exception):
            print(f"Fitting {name}...")
            model.fit(**params)
            model.save_model(output_folder, file_name=file_name)
            print(f"Saved {name} to disk.")
        #print(type(recommender_slim.W_sparse), recommender_slim.W_sparse.nnz)

    # 4. Popolamento lista per ottimizzazione/test
    fold_data = {
        "URM_train": URM_train,
        "slim": recommender_slim,
        "ease": recommender_ease,
        "rp3": recommender_rp3,
        "evaluator": evaluator_test
    }
    
    prefitted_folds.append(fold_data)

print("\nTask completato: tutti i fold sono pronti in memoria.")

print("Pre-training completato.")

In [ ]:

import time 

class SaveResults(object):
    
    def __init__(self):
        self.results_df = pd.DataFrame(columns=["result", "train_time (min)"])
    
    def __call__(self, optuna_study, optuna_trial):
        hyperparam_dict = optuna_trial.params.copy()
        hyperparam_dict["result"] = optuna_trial.values[0]
        
        # Retrieve the optimal number of epochs and training time from the "user attributes" of the trial
        #hyperparam_dict["epochs"] = optuna_trial.user_attrs["epochs"]
        hyperparam_dict["train_time (min)"] = optuna_trial.user_attrs["train_time (min)"]
        
        self.results_df.loc[len(self.results_df)] = hyperparam_dict
        
        
def objective_function_funksvd(optuna_trial):

    start_time = time.time()
    scores = []
    for fold_data in prefitted_folds:
        
        # Recuperiamo i modelli e i dati dal dizionario
        URM_train = fold_data["URM_train"]
        recommender_slim = fold_data["slim"]
        recommender_ease = fold_data["ease"]
        recommender_rp3 = fold_data["rp3"]
        evaluator_test = fold_data["evaluator"]
        
        
        recommender = SimilarityMergingHybridRecommender(
            URM_train, 
            recommender_slim, 
            recommender_ease,
            recommender_rp3
        )
        
        alpha=optuna_trial.suggest_float("alpha", 0.25, 0.4)
        beta=optuna_trial.suggest_float("beta", 0.0, 0.15)
        
        recommender.fit(alpha, beta)
        
        result, _ = evaluator_test.evaluateRecommender(recommender)
        #print("prova = ", result["MAP"].values[0])
        #print(i)
        #print(result["RECALL"])
        scores.append(result["RECALL"].values[0])
        #if result["MAP"].values[0] < 0.051:
        #    break
        
    # Add the number of epochs selected by earlystopping as a "user attribute" of the optuna trial
    #epochs = recommender_instance.get_early_stopping_final_epochs_dict()["epochs"]
    #optuna_trial.set_user_attr("epochs", epochs) 
    optuna_trial.set_user_attr("train_time (min)", (time.time() - start_time)/60) 
    print(scores)
    return sum(scores) / len(scores)


In [ ]:
import optuna

optuna_study = optuna.create_study(direction="maximize")
        
save_results = SaveResults()
        
optuna_study.optimize(objective_function_funksvd,
                      callbacks=[save_results],
                      n_trials = 100)

# Da qui inizia il training su URM_all

In [ ]:
return

In [ ]:
best_alpha_test = 0.9874643151475879
best_beta_test = 0.8761805316137236
#Trial 188 finished with value: 0.2909859712486159 and parameters: {'alpha': 0.9874643151475879, 'beta': 0.8761805316137236}.

In [ ]:
recommender_knn_f = ItemKNNCFRecommender(URM_all)
recommender_knn_f.fit(**KNN_params)

In [ ]:
recommender_SLIM_f = SLIMElasticNetRecommender(URM_all)
recommender_SLIM_f.fit(**SLIM_params)

In [ ]:
# Train the final model 
new_similarity = (1 - best_alpha_test) * csr_matrix(recommender_knn_f.W_sparse) + best_alpha_test * recommender_SLIM_f.W_sparse 

hybridrecommender_object_f = ItemKNNCustomSimilarityRecommender(URM_all)
hybridrecommender_object_f.fit(new_similarity)

In [ ]:
als_recommender_f = FeatureCombinedImplicitALSRecommender(URM_all)
als_recommender_f.fit(**IALS_params)

In [ ]:
recommenders = [hybridrecommender_object_f, als_recommender_f]

linear_comb_rec_f = GeneralizedLinearCoupleHybridRecommender(URM_all, recommenders)
linear_comb_rec_f.fit(best_beta_test)

In [ ]:
import time

start_time = time.time()
results = []
for user_id in df_test_user["user_id"].tolist():
        recommendations = linear_comb_rec_f.recommend(user_id, cutoff=20)       
       
        results.append((user_id, ' '.join(map(str, recommendations))))

df_recommendations = pd.DataFrame(results, columns=["user_id", "item_list"])
print(df_recommendations)
df_recommendations.to_csv("recommendations_m2_knn_1.csv", index=False)

end_time = time.time()